# Phase 2.3 — EDA et vérification de l'équilibre au baseline

**Projet** : RCT Foyers Améliorés — Sénégal rural  
**Notebook** : `python/01_eda.ipynb`

Dans un RCT, l'EDA a un rôle précis : **vérifier que la randomisation a bien équilibré les bras** sur les caractéristiques pré-traitement, et documenter la composition de l'échantillon. Il ne s'agit pas de diagnostiquer un biais de sélection (la randomisation l'empêche en espérance) mais de s'en assurer empiriquement et de détecter tout déséquilibre résiduel à signaler.

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'python' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
FIG  = ROOT / 'docs' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / 'panel_ancova.parquet')
print(f"Panel : {df.shape[0]} ménages × {df.shape[1]} variables")
print(f"Bras  : {df['treatment'].value_counts().to_dict()}")

# Labels pour l'affichage
ARM_LABELS = {'T1': 'T1 - Foyer + formation',
              'T2': 'T2 - Formation seule',
              'T3': 'T3 - Controle'}
ARM_COLORS = {'T1': '#d62728', 'T2': '#1f77b4', 'T3': '#7f7f7f'}
df['arm_label'] = df['treatment'].map(ARM_LABELS)

## 1. Composition de l'échantillon

In [ ]:
# Distribution par bras, région et zone
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bras
ax = axes[0]
counts = df['treatment'].value_counts()[['T1','T2','T3']]
bars = ax.bar(counts.index, counts.values,
              color=[ARM_COLORS[t] for t in counts.index], width=0.55)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', fontsize=11, fontweight='bold')
ax.set_title('Par bras de traitement', fontweight='bold')
ax.set_ylim(0, max(counts)*1.15)
ax.set_ylabel('Nombre de ménages')

# Région
ax = axes[1]
reg = df.groupby('region')['treatment'].value_counts().unstack(fill_value=0)[['T1','T2','T3']]
reg.plot(kind='barh', ax=ax,
         color=[ARM_COLORS['T1'], ARM_COLORS['T2'], ARM_COLORS['T3']],
         width=0.7)
ax.set_title('Par région', fontweight='bold')
ax.set_xlabel('Ménages')
ax.legend(title='Bras', fontsize=9)

# Zone agro-écologique
ax = axes[2]
zone = df.groupby('zone')['treatment'].value_counts().unstack(fill_value=0)[['T1','T2','T3']]
zone.plot(kind='barh', ax=ax,
          color=[ARM_COLORS['T1'], ARM_COLORS['T2'], ARM_COLORS['T3']],
          width=0.7)
ax.set_title('Par zone agro-écologique (strate)', fontweight='bold')
ax.set_xlabel('Ménages')
ax.legend(title='Bras', fontsize=9)

plt.suptitle("Composition de l'échantillon", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'sample_composition.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nÉquilibre par zone :")
print(zone.to_string())

## 2. Analyse de l'attrition

1 ménage perdu sur 1 851 (0,05 %) — attrition quasi-nulle. On vérifie qu'elle n'est pas différentielle par bras.

In [ ]:
# Baseline total par bras (avant inner join)
# On relit les téléphones baseline pour connaître le N avant matching
import hashlib, os
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')
SALT = os.getenv('ANON_SALT', '')

b_phones = pd.read_csv(
    next(ROOT.glob('data/raw/*Baseline*')),
    usecols=['tel_cible', 'treatment'], low_memory=False
)
def norm(s):
    s = ''.join(c for c in str(s) if c.isdigit())
    if s.startswith('221') and len(s) == 12: s = s[3:]
    return s if len(s) >= 7 else None

b_phones['_phone'] = b_phones['tel_cible'].apply(norm)
junk = {'888','999','777777777','7777777777','0',''}
b_phones = b_phones[~b_phones['_phone'].isin(junk) & b_phones['_phone'].notna()]
b_phones = b_phones.sample(frac=1, random_state=42).drop_duplicates('_phone', keep='first')

print("=== Attrition par bras ===")
base_n = b_phones['treatment'].value_counts()
panel_n = df['treatment'].value_counts()
att = pd.DataFrame({'N baseline': base_n, 'N panel': panel_n})
att['Perdus'] = att['N baseline'] - att['N panel']
att['Taux (%)'] = (att['Perdus'] / att['N baseline'] * 100).round(2)
print(att.to_string())

# Test de Fisher : attrition indépendante du bras ?
from scipy.stats import chi2_contingency
ct = pd.crosstab(b_phones['treatment'],
                 b_phones['_phone'].isin(df['hh_id'].apply(
                     lambda h: h  # hh_id deja hache, comparaison directe
                 )))
# Note : avec 1 seule observation attrition, le test est non informatif
print(f"\nTest d'indépendance attrition × bras : non informatif (n=1 perdu)")
print("Conclusion : attrition négligeable, aucun risque de biais.")

## 3. Table d'équilibre au baseline

**Le test clé du RCT.** Pour chaque variable pré-traitement, on vérifie que les moyennes des trois bras sont comparables. On reporte la moyenne par bras + p-valeur du test F (ANOVA à un facteur pour les continues, χ² pour les binaires/catégorielles).

In [ ]:
# Tailles d'échantillon fixes (identiques pour toutes les variables)
N = {arm: int((df['treatment']==arm).sum()) for arm in ['T1','T2','T3']}
COL = {arm: f"{arm} (n={N[arm]})" for arm in ['T1','T2','T3']}

def balance_row(varname, label, data=df, var_type='continuous'):
    sub = data[['treatment', varname]].dropna()
    groups = [sub.loc[sub['treatment']==arm, varname].values for arm in ['T1','T2','T3']]
    means = [g.mean() for g in groups]

    if var_type == 'continuous':
        f_stat, p_val = stats.f_oneway(*groups)
    else:
        ct = pd.crosstab(sub[varname], sub['treatment'])
        _, p_val, _, _ = stats.chi2_contingency(ct)

    return {
        'Variable'  : label,
        COL['T1']   : round(means[0], 3),
        COL['T2']   : round(means[1], 3),
        COL['T3']   : round(means[2], 3),
        'p-valeur'  : round(p_val, 3),
        'Equilibre' : 'oui' if p_val > 0.05 else 'NON',
    }

balance_vars = [
    ('taille_menage_B',              'Taille du menage',                   'continuous'),
    ('taches_menage_h_mean_B',       'Taches menageres / jour (h)',         'continuous'),
    ('travail_propre_compte_h_mean_B','Travail propre compte / jour (h)',   'continuous'),
    ('revenu_femme_B',               'Revenu mensuel femme ciblee (FCFA)',  'continuous'),
    ('revenu_menage_B',              'Revenu mensuel menage (FCFA)',        'continuous'),
    ('score_O_global_B',             'Score attitude entrepreneuriale O',   'continuous'),
    ('score_entrepreneurial_B',      'Score motivation travail (O1-O9)',    'continuous'),
    ('score_genre_attitudes_B',      'Score attitudes genre (O19-O30)',     'continuous'),
    ('score_autonomie_B',            'Score autonomisation (actif08-16)',   'continuous'),
    ('D56_charbon_B',                'Stock charbon (FCFA)',                'continuous'),
    ('D56_bois_B',                   'Stock bois (FCFA)',                   'continuous'),
    ('N17_B',                        'Echelle de bien-etre (1-6)',          'continuous'),
]

rows = []
for var, label, vtype in balance_vars:
    if var in df.columns:
        rows.append(balance_row(var, label, var_type=vtype))
    else:
        print(f"Absent : {var}")

bal_table = pd.DataFrame(rows)
print("=== Table d'equilibre au baseline ===")
print(bal_table.to_string(index=False))

n_unbal = (bal_table['p-valeur'] < 0.05).sum()
print(f"\nVariables desequilibrees (p<0.05) : {n_unbal}/{len(bal_table)}")
if n_unbal == 0:
    print("-> Equilibre excellent - coherent avec une randomisation par grappes reussie.")
elif n_unbal <= 2:
    print("-> Desequilibre marginal - a controler comme covariable dans l\'ANCOVA.")
else:
    print("-> Desequilibre notable - verifier la procedure de randomisation.")

In [ ]:
# Sauvegarder la table d'équilibre
bal_table.to_csv(FIG / 'balance_table.csv', index=False, encoding='utf-8')

# Visualisation : balance plot (differences normalisees)
fig, ax = plt.subplots(figsize=(10, 5))
vars_short = [v.replace(' (FCFA)', '').replace(' (h)', '')
                .replace('Score ', '').replace('Taches menageres', 'Taches men.')
              for v in bal_table['Variable']]

# Difference T1-T3 et T2-T3 normalisees par SD poolee
for arm, color, label in [('T1', '#d62728', 'T1 vs T3'), ('T2', '#1f77b4', 'T2 vs T3')]:
    diffs = []
    for _, row in bal_table.iterrows():
        d = row[f'{arm} (n={len(df[df["treatment"]==arm])})'] -             row[f'T3 (n={len(df[df["treatment"]=="T3"])})']
        # Normaliser par l'écart-type global
        var_name = [v for v, lbl, _ in balance_vars if lbl == row['Variable']]
        if var_name and var_name[0] in df.columns:
            sd = df[var_name[0]].std()
            diffs.append(d / sd if sd > 0 else 0)
        else:
            diffs.append(0)
    offset = 0.15 if arm == 'T1' else -0.15
    ax.scatter(diffs, [i + offset for i in range(len(diffs))],
               color=color, label=label, s=60, zorder=3)

ax.axvline(0, color='black', lw=0.8)
ax.axvline(0.1, color='orange', lw=0.6, ls='--', alpha=0.7)
ax.axvline(-0.1, color='orange', lw=0.6, ls='--', alpha=0.7)
ax.axvline(0.2, color='red', lw=0.6, ls='--', alpha=0.5)
ax.axvline(-0.2, color='red', lw=0.6, ls='--', alpha=0.5)
ax.set_yticks(range(len(vars_short)))
ax.set_yticklabels(vars_short)
ax.set_xlabel('Difference standardisee vs controle (T3)')
ax.set_title("Balance au baseline\n(differences standardisees vs T3)", fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIG / 'balance_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Distribution des outcomes au baseline par bras

In [ ]:
# Outcomes continus : boxplots par bras
outcome_pairs = [
    ('taches_menage_h_mean_B',  'Taches menageres / jour (h)'),
    ('revenu_femme_B',          'Revenu femme ciblee (FCFA)'),
    ('score_O_global_B',        'Score attitude entrepreneuriale'),
    ('score_autonomie_B',       'Score autonomisation'),
    ('N17_B',                   'Bien-être (1-6)'),
    ('D56_charbon_B',           'Stock charbon (FCFA)'),
]
outcome_pairs = [(v, l) for v, l in outcome_pairs if v in df.columns]

ncols = 3
nrows = (len(outcome_pairs) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows))

df_plot = df.copy()
df_plot['arm'] = df_plot['treatment'].map(ARM_LABELS)

for ax, (var, label) in zip(axes.flat, outcome_pairs):
    sub = df_plot[['arm', var]].dropna()
    sns.boxplot(data=sub, x='arm', y=var, ax=ax,
                hue = 'arm', legend = False,
                palette=[ARM_COLORS['T1'], ARM_COLORS['T2'], ARM_COLORS['T3']],
                width=0.5, fliersize=2)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelrotation=15, labelsize=8)

for ax in axes.flat[len(outcome_pairs):]:
    ax.set_visible(False)

plt.suptitle('Distribution des outcomes au baseline par bras de traitement',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG / 'baseline_outcomes_by_arm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Score O : distribution détaillée (c'est le plus directement lié à la formation)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (arm, label, color) in zip(axes, [
    ('T1', 'T1 - Foyer + formation', ARM_COLORS['T1']),
    ('T2', 'T2 - Formation seule',   ARM_COLORS['T2']),
    ('T3', 'T3 - Controle',          ARM_COLORS['T3']),
]):
    sub = df.loc[df['treatment']==arm, 'score_O_global_B'].dropna()
    ax.hist(sub, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(sub.mean(), color='black', ls='--', lw=1.2,
               label=f'Moy. = {sub.mean():.2f}')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Score attitude entrepreneuriale (O1-O30)')
    ax.set_ylabel('N ménages' if ax == axes[0] else '')
    ax.legend(fontsize=9)

plt.suptitle('Distribution du score O au baseline - distribution symetrique attendue',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'score_O_baseline_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Complétude des données

In [ ]:
# Taux de complétude par variable et par bras
key_outcomes = [
    'taches_menage_h_mean_B', 'taches_menage_h_mean_E',
    'revenu_femme_B', 'revenu_femme_E',
    'revenu_menage_B', 'revenu_menage_E',
    'score_O_global_B', 'score_O_global_E',
    'score_autonomie_B', 'score_autonomie_E',
    'N17_B', 'N17_E',
    'D56_charbon_B', 'D56_charbon_E',
    'foyer_cuisson_h_E', 'foyer_frequence_E',
]
key_outcomes = [c for c in key_outcomes if c in df.columns]

complet = df.groupby('treatment')[key_outcomes].apply(
    lambda d: d.notna().mean() * 100
).T
complet.columns = ['T1 (%)', 'T2 (%)', 'T3 (%)']
complet['Moy. (%)'] = complet.mean(axis=1).round(1)
complet = complet.round(1)

print("=== Complétude des outcomes clés (%) ===")
print(complet.to_string())

# Alerte pour variables < 70%
low_cov = complet[complet['Moy. (%)'] < 70]
if len(low_cov):
    print(f"\n!! Variables avec completude < 70% :")
    print(low_cov[['Moy. (%)']].to_string())
else:
    print("\nOK - Toutes les variables ont une completude >= 70%")

## 6. Synthèse et orientations pour la Phase 2.4

Documenter ici après lecture des résultats :

**Équilibre au baseline** — si tous les p-valeurs > 0.05 et toutes les différences standardisées < 0.10, la randomisation est validée : les estimations ANCOVA peuvent s'appuyer sur une comparaison directe (le contrôle des valeurs baseline est une amélioration de précision, pas une correction de biais).

**Variables à contrôler dans l'ANCOVA** — même avec une randomisation parfaite, contrôler les valeurs baseline des outcomes améliore la puissance statistique. Pour chaque question Q1-Q4, l'ANCOVA inclura :  
`Y_E = α + β₁·T1 + β₂·T2 + γ·Y_B + δ·zone + ε`

**Outcomes disponibles confirmés** :
- Q1 (temps) : `taches_menage_h_mean_B/_E` (≥93%), `foyer_cuisson_h_E` (endline T1)
- Q2 (revenu) : `revenu_femme_B/_E`, `revenu_menage_B/_E`
- Q3 (attitude) : `score_O_global_B/_E`, `score_entrepreneurial_B/_E`
- Q4 (combustibles) : `D56_charbon_B/_E`, `N17_B/_E`